# 00 — Validación y extracción de datos

## Objetivo
Verificar esquema, integridad, fechas, signos y relaciones con catálogos antes del análisis. El archivo fuente no se modifica.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_catalogs, load_transactions, validate_transactions

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
transactions = load_transactions(ROOT / "transacciones_sinteticas.csv")
types, banks = load_catalogs(
    ROOT / "catalogo_tipos_subtipos.csv", ROOT / "catalogo_bancos.csv"
)
transactions.shape, types.shape, banks.shape

## Esquema y catálogos
Los códigos son categorías y se usan para validar relaciones, no como variables continuas.

In [ ]:
print(transactions.schema)
print(types.schema)
print(banks.schema)
report = validate_transactions(transactions, types, banks)
for key, value in report.items():
    print(f"{key}: {value}")

## Criterios de aceptación
La naturaleza del movimiento se valida con `tipo_movimiento`, además del signo del monto. Los hallazgos no se corrigen silenciosamente.

In [ ]:
assert report["invalid_catalog_pairs"] == 0
assert report["sign_mismatches"] == 0
assert report["invalid_dates"] == 0
assert report["catalog_duplicate_pairs"] == 0
assert report["observed_transaction_types"] == report["catalog_transaction_types"] == 57
assert report["unique_clients"] == 10_000
print("Validación estructural y semántica aprobada.")

## Decisión para la siguiente etapa
Con estas comprobaciones aprobadas, el dataset puede pasar al EDA y a la construcción de características agregadas por cliente.